# PW02 · Source Merge And Global Thresholds

用途：在 Colab 冷启动环境中合并 PW01 的 formal 双 role shard（positive_source 与 clean_negative），并在存在完整 planner_conditioned_control_negative cohort 时额外导出 optional diagnostic pool；随后构建 content 与 attestation 两条全局阈值链，并回读 PW02 summary、top-level finalize manifest 与诚实的 system_final derived metrics。

## 1) 定义路径与参数

本单元只做常量定义与最小工具函数准备，不执行任何外部副作用。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW02_Source_Merge_And_Global_Thresholds"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
LOCAL_RUNTIME_ENABLED = True
LOCAL_PROJECT_ROOT = Path("/content/CEG_WM_PaperWorkflow")
PERSISTENT_DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
DRIVE_BUNDLE_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow_Bundles"
if LOCAL_RUNTIME_ENABLED:
    DRIVE_PROJECT_ROOT = LOCAL_PROJECT_ROOT
else:
    DRIVE_PROJECT_ROOT = PERSISTENT_DRIVE_PROJECT_ROOT
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW02_Source_Merge_And_Global_Thresholds.py"
DRIVE_MODELS_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "Models"
PERSISTENT_INSPYRENET_ROOT = DRIVE_MODELS_ROOT / "inspyrenet"
PERSISTENT_HF_ROOT = DRIVE_MODELS_ROOT / "Huggingface"
LOCAL_HF_HOME = REPO_ROOT / "huggingface_cache"
LOCAL_HF_HUB_CACHE = LOCAL_HF_HOME / "hub"
LOCAL_TRANSFORMERS_CACHE = LOCAL_HF_HOME / "transformers"

FAMILY_ID = "paper_eval_family_geometry_interval_discovery_v2"

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 2) 挂载 Drive 并同步仓库

本单元保证 PW02 可以在空白 Colab 会话中直接冷启动。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from paper_workflow.scripts.pw_local_runtime import prepare_local_runtime_for_stage
from scripts.notebook_runtime_common import resolve_notebook_model_cache_layout

if LOCAL_RUNTIME_ENABLED:
    prepare_local_runtime_for_stage(
        stage_name="PW02_Source_Merge_And_Global_Thresholds",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        clean_before_run=True,
        include_optional_control_negative=False,
    )

MODEL_CACHE_LAYOUT = resolve_notebook_model_cache_layout(DRIVE_MOUNT_ROOT, REPO_ROOT, create_directories=True)
DRIVE_MODELS_ROOT = MODEL_CACHE_LAYOUT["drive_models_root"]
PERSISTENT_INSPYRENET_ROOT = MODEL_CACHE_LAYOUT["persistent_inspyrenet_root"]
PERSISTENT_HF_ROOT = MODEL_CACHE_LAYOUT["persistent_hf_root"]
LOCAL_HF_HOME = MODEL_CACHE_LAYOUT["local_hf_home"]
LOCAL_HF_HUB_CACHE = MODEL_CACHE_LAYOUT["local_hf_hub_cache"]
LOCAL_TRANSFORMERS_CACHE = MODEL_CACHE_LAYOUT["local_transformers_cache"]

os.environ["HF_HOME"] = str(LOCAL_HF_HOME)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(LOCAL_HF_HUB_CACHE)
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_TRANSFORMERS_CACHE)

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
        "drive_models_root": str(DRIVE_MODELS_ROOT),
        "persistent_inspyrenet_root": str(PERSISTENT_INSPYRENET_ROOT),
        "persistent_hf_root": str(PERSISTENT_HF_ROOT),
        "local_hf_home": str(LOCAL_HF_HOME),
    },
)

## 3) 运行前 precheck

本单元只把 positive_source 与 clean_negative 视为 PW02 formal 主流程硬依赖；planner_conditioned_control_negative 缺失可接受，但若只完成部分 shard，PW02 CLI 会快速失败。

In [ ]:
FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
SOURCE_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_shard_plan.json"
SOURCE_SPLIT_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_split_plan.json"
PW02_SUMMARY_PATH = FAMILY_ROOT / "runtime_state" / "pw02_summary.json"
PROJECT_ROOT_PRECHECK_LABEL = "项目运行根目录存在" if LOCAL_RUNTIME_ENABLED else "Drive 项目根目录存在"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck(PROJECT_ROOT_PRECHECK_LABEL, DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("Drive 模型根目录存在", DRIVE_MODELS_ROOT.exists(), str(DRIVE_MODELS_ROOT))
record_precheck("persistent InSPyReNet 目录存在", PERSISTENT_INSPYRENET_ROOT.exists(), str(PERSISTENT_INSPYRENET_ROOT))
record_precheck("persistent Huggingface 目录存在", PERSISTENT_HF_ROOT.exists(), str(PERSISTENT_HF_ROOT))
record_precheck("本地 HF_HOME 目录存在", LOCAL_HF_HOME.exists(), str(LOCAL_HF_HOME))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW02 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("source shard plan 存在", SOURCE_SHARD_PLAN_PATH.exists(), str(SOURCE_SHARD_PLAN_PATH))
record_precheck("source split plan 存在", SOURCE_SPLIT_PLAN_PATH.exists(), str(SOURCE_SPLIT_PLAN_PATH))
record_precheck("positive shard 根目录存在", (FAMILY_ROOT / "source_shards" / "positive").exists(), str(FAMILY_ROOT / "source_shards" / "positive"))
record_precheck("negative shard 根目录存在", (FAMILY_ROOT / "source_shards" / "negative").exists(), str(FAMILY_ROOT / "source_shards" / "negative"))

print_json(
    "pw02_precheck",
    {
        "model_cache_layout": {
            "drive_models_root": str(DRIVE_MODELS_ROOT),
            "persistent_inspyrenet_root": str(PERSISTENT_INSPYRENET_ROOT),
            "persistent_hf_root": str(PERSISTENT_HF_ROOT),
            "local_hf_home": str(LOCAL_HF_HOME),
            "local_hf_hub_cache": str(LOCAL_HF_HUB_CACHE),
            "local_transformers_cache": str(LOCAL_TRANSFORMERS_CACHE),
        },
        "items": PRECHECK_RESULTS,
    },
)

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW02 precheck failed: {hard_fail}")

## 4) 执行 PW02 CLI

本单元调用 paper_workflow/scripts/PW02_Source_Merge_And_Global_Thresholds.py，并回读标准 JSON summary。summary 会显式记录 planner_conditioned_control_negative 的 cohort_status。

In [ ]:
from datetime import datetime, timezone
import time

from paper_workflow.scripts.pw_local_runtime import archive_local_runtime_for_stage
from scripts.notebook_runtime_common import (
    build_repo_import_subprocess_env,
    build_stage_runtime_diagnostics_payload,
    build_stage_runtime_workload_summary,
    write_stage_runtime_diagnostics,
)

COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
]

PW02_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)
PW02_RUNTIME_DIAGNOSTICS_PATH = PW02_SUMMARY_PATH.parent / "pw02_runtime_diagnostics.json"
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_STARTED_AT_MONOTONIC = time.perf_counter()
PW02_RESULT = subprocess.run(
    COMMAND,
    cwd=REPO_ROOT,
    env=PW02_SUBPROCESS_ENV,
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
RUN_FINISHED_AT_UTC = datetime.now(timezone.utc).isoformat()
RUN_ELAPSED_SECONDS = time.perf_counter() - RUN_STARTED_AT_MONOTONIC

if PW02_RESULT.returncode != 0:
    raise RuntimeError(
        "PW02 failed: "
        f"return_code={PW02_RESULT.returncode} stdout={PW02_RESULT.stdout} stderr={PW02_RESULT.stderr}"
    )

if not PW02_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        "PW02 summary file missing after successful subprocess execution: "
        f"summary_path={PW02_SUMMARY_PATH} stdout={PW02_RESULT.stdout} stderr={PW02_RESULT.stderr}"
    )

PW02_SUMMARY = json.loads(PW02_SUMMARY_PATH.read_text(encoding="utf-8"))
print_json("pw02_summary", PW02_SUMMARY)
PW02_SYSTEM_FINAL_METRICS = (
    dict(PW02_SUMMARY.get("system_final_metrics", {}))
    if isinstance(PW02_SUMMARY.get("system_final_metrics"), dict)
    else {}
)
PW02_SPLIT_COUNTS = (
    dict(PW02_SUMMARY.get("split_counts", {}))
    if isinstance(PW02_SUMMARY.get("split_counts"), dict)
    else {}
)

def runtime_count(value):
    return int(value) if isinstance(value, int) and not isinstance(value, bool) and value >= 0 else 0

PW02_COUNT_SUMMARY = {
    "calibration_split_count": runtime_count(PW02_SPLIT_COUNTS.get("calibration")),
    "evaluate_split_count": runtime_count(PW02_SPLIT_COUNTS.get("evaluate")),
    "n_positive": runtime_count(PW02_SYSTEM_FINAL_METRICS.get("n_positive")),
    "n_negative": runtime_count(PW02_SYSTEM_FINAL_METRICS.get("n_negative")),
    "n_total": runtime_count(PW02_SYSTEM_FINAL_METRICS.get("n_total")),
}
PW02_WORKLOAD_UNIT_COUNT = PW02_COUNT_SUMMARY["n_total"]
if PW02_WORKLOAD_UNIT_COUNT == 0:
    PW02_WORKLOAD_UNIT_COUNT = PW02_COUNT_SUMMARY["evaluate_split_count"]
PW02_RUNTIME_DIAGNOSTICS = build_stage_runtime_diagnostics_payload(
    stage_name="PW02_Source_Merge_And_Global_Thresholds",
    family_id=FAMILY_ID,
    expected_output_label="pw02_summary",
    expected_output_path=PW02_SUMMARY_PATH,
    started_at_utc=RUN_STARTED_AT_UTC,
    finished_at_utc=RUN_FINISHED_AT_UTC,
    elapsed_seconds=RUN_ELAPSED_SECONDS,
    return_code=PW02_RESULT.returncode,
    stdout_text=PW02_RESULT.stdout,
    stderr_text=PW02_RESULT.stderr,
    count_summary=PW02_COUNT_SUMMARY,
    workload_summary=build_stage_runtime_workload_summary(
        unit_label="evaluated_events",
        unit_count=PW02_WORKLOAD_UNIT_COUNT,
        elapsed_seconds=RUN_ELAPSED_SECONDS,
    ),
)
write_stage_runtime_diagnostics(
    diagnostics_path=PW02_RUNTIME_DIAGNOSTICS_PATH,
    payload=PW02_RUNTIME_DIAGNOSTICS,
)
print_json("pw02_runtime_diagnostics", PW02_RUNTIME_DIAGNOSTICS)

pw02_finalize_manifest_path_value = PW02_SUMMARY.get("paper_source_finalize_manifest_path")
if isinstance(pw02_finalize_manifest_path_value, str) and pw02_finalize_manifest_path_value:
    PW02_FINALIZE_MANIFEST_PATH = Path(pw02_finalize_manifest_path_value)
    if PW02_FINALIZE_MANIFEST_PATH.exists():
        PW02_FINALIZE_MANIFEST = json.loads(PW02_FINALIZE_MANIFEST_PATH.read_text(encoding="utf-8"))
        print_json("pw02_finalize_manifest", PW02_FINALIZE_MANIFEST)

system_final_metrics_artifact_path_value = PW02_SUMMARY.get("system_final_metrics_artifact_path")
if isinstance(system_final_metrics_artifact_path_value, str) and system_final_metrics_artifact_path_value:
    SYSTEM_FINAL_METRICS_ARTIFACT_PATH = Path(system_final_metrics_artifact_path_value)
    if SYSTEM_FINAL_METRICS_ARTIFACT_PATH.exists():
        PW02_SYSTEM_FINAL_METRICS = json.loads(SYSTEM_FINAL_METRICS_ARTIFACT_PATH.read_text(encoding="utf-8"))
        print_json("pw02_system_final_metrics", PW02_SYSTEM_FINAL_METRICS)

if LOCAL_RUNTIME_ENABLED:
    archive_local_runtime_for_stage(
        stage_name="PW02_Source_Merge_And_Global_Thresholds",
        family_id=FAMILY_ID,
        local_project_root=LOCAL_PROJECT_ROOT,
        drive_bundle_root=DRIVE_BUNDLE_ROOT,
        clean_after_archive=False,
    )

## 5) 关键结果摘要

In [ ]:
formal_metrics = PW02_SUMMARY.get("formal_final_decision_metrics", {}) or {}
system_union_metrics = PW02_SUMMARY.get("derived_system_union_metrics", {}) or {}

pw02_key_outputs = {
    "family_id": PW02_SUMMARY.get("family_id"),
    "pw02_stage_root": PW02_SUMMARY.get("pw02_stage_root"),
    "paper_source_finalize_manifest_path": PW02_SUMMARY.get("paper_source_finalize_manifest_path"),
    "formal_final_decision": {
        "n_positive": formal_metrics.get("n_positive"),
        "n_negative": formal_metrics.get("n_negative"),
        "n_total": formal_metrics.get("n_total"),
        "final_decision_available_rate": formal_metrics.get("final_decision_available_rate"),
        "final_decision_tpr": formal_metrics.get("final_decision_tpr"),
        "final_decision_fpr": formal_metrics.get("final_decision_fpr"),
        "final_decision_status_counts": formal_metrics.get("final_decision_status_counts"),
    },
    "system_union_check": {
        "n_positive": system_union_metrics.get("n_positive"),
        "n_negative": system_union_metrics.get("n_negative"),
        "n_total": system_union_metrics.get("n_total"),
        "system_tpr": system_union_metrics.get("system_tpr"),
        "system_fpr": system_union_metrics.get("system_fpr"),
        "event_attestation_tpr": system_union_metrics.get("event_attestation_tpr"),
        "event_attestation_fpr": system_union_metrics.get("event_attestation_fpr"),
        "geo_rescue_eligible_rate": system_union_metrics.get("geo_rescue_eligible_rate"),
        "geo_rescue_applied_rate": system_union_metrics.get("geo_rescue_applied_rate"),
    },
    "optional_control_negative": {
        "role_requirement": PW02_SUMMARY.get("planner_conditioned_control_negative_role_requirement"),
        "cohort_status": PW02_SUMMARY.get("planner_conditioned_control_negative_cohort_status"),
        "discovered_source_shard_count": PW02_SUMMARY.get("planner_conditioned_control_negative_discovered_source_shard_count"),
        "expected_source_shard_count": PW02_SUMMARY.get("planner_conditioned_control_negative_expected_source_shard_count"),
        "interpretation": "not_required_for_current_pilot",
    },
    "key_artifacts_for_next_stages": {
        "positive_source_pool_manifest_path": PW02_SUMMARY.get("positive_source_pool_manifest_path"),
        "clean_negative_pool_manifest_path": PW02_SUMMARY.get("clean_negative_pool_manifest_path"),
        "clean_event_table_path": PW02_SUMMARY.get("clean_event_table_path"),
        "payload_clean_summary_path": PW02_SUMMARY.get("payload_clean_summary_path"),
        "clean_quality_pair_manifest_path": PW02_SUMMARY.get("clean_quality_pair_manifest_path"),
        "formal_final_decision_metrics_artifact_path": PW02_SUMMARY.get("formal_final_decision_metrics_artifact_path"),
        "derived_system_union_metrics_artifact_path": PW02_SUMMARY.get("derived_system_union_metrics_artifact_path"),
    },
}

print_json("PW02 关键结果摘要", pw02_key_outputs)